# Phase 12: Parallel Training - FOLD 4 (TARGETED)

**Objective:** Complete 5-Fold Training with **Targeted Fine-Tuning** on Fold 4.

In [ ]:
# Cell 1: Mount Drive & Clone Repo
from google.colab import drive
import subprocess, sys, os, shutil

drive.mount('/content/drive')
print('✅ Drive mounted')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'scikit-learn'], timeout=120)
REPO_DIR = '/content/phase2'
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/nithin12342/phase2.git', REPO_DIR], timeout=120, check=True)
PROJECT_ROOT = os.path.join(REPO_DIR, 'ml_pipeline', 'h5_omnifusion')
if PROJECT_ROOT not in sys.path: sys.path.insert(0, PROJECT_ROOT)
import torch, numpy as np, pandas as pd

In [ ]:
# Cell 2: Setup
root_dir = '/content/drive/MyDrive/DAIC-WOZ_Datasets'
H5_ROOT = os.path.join(root_dir, 'H5_OmniFusion_Output')
MERGED_CSV = os.path.join(root_dir, 'merged_all_labels.csv')
CHECKPOINT_DIR = os.path.join(root_dir, 'checkpoints_phase10_finetune')
SAVE_DIR = os.path.join(root_dir, 'checkpoints_phase12')
ACHIEVED_DIR = os.path.join(root_dir, 'achieved')
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(ACHIEVED_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FOLD_TO_RUN = 4

In [ ]:
# Cell 3: STAGE 1 - BROAD
from src.models.h5_omnifusion import H5OmniFusion
from config.model_config import H5Config, ComputeTier
from src.data.h5_dataset import create_h5_dataloaders_kfold
from src.training.trainer import H5Trainer
from config.training_config import TrainingConfig

fold = FOLD_TO_RUN
train_loader, val_loader, test_loader = create_h5_dataloaders_kfold(
    h5_dir=H5_ROOT, labels_csv=MERGED_CSV, fold_idx=fold, n_folds=5, batch_size=16, num_workers=2
)

model = H5OmniFusion(config=H5Config.from_tier(ComputeTier.MEDIUM))
ckpt_path = os.path.join(CHECKPOINT_DIR, f'h5_omnifusion_medium_fold{fold}_best.pt')
if not os.path.exists(ckpt_path): ckpt_path = os.path.join(CHECKPOINT_DIR, 'h5_omnifusion_medium_fold4_best.pt')
if os.path.exists(ckpt_path): model.load_state_dict(torch.load(ckpt_path, map_location='cpu', weights_only=False), strict=False)

model = model.to(DEVICE)
t_config = TrainingConfig()
t_config.batch_size, t_config.n_epochs = 16, 15
t_config.loss.focal_alpha, t_config.loss.lambda_phq = 0.50, 0.2

trainer = H5Trainer(model, train_loader, val_loader, t_config, test_loader, DEVICE)
trainer.train(save_path=os.path.join(SAVE_DIR, f'fold{fold}_phase12_stage1.pt'))

In [ ]:
# Cell 4: STAGE 2 - TARGETED
print(f'\n🚀 STAGE 2: Targeted Fine-Tuning')
daic_pids = [p for p in train_loader.dataset.participant_ids if not str(p).startswith(('t_', 'lmvd_'))]
from src.data.h5_dataset import H5OmniFusionDataset
from torch.utils.data import DataLoader
target_loader = DataLoader(H5OmniFusionDataset(H5_ROOT, participant_ids=daic_pids, labels_csv=MERGED_CSV), batch_size=8, shuffle=True)

t_config.optimizer.lr, t_config.n_epochs = 2e-6, 5
trainer.train_loader = target_loader
trainer.train(save_path=os.path.join(SAVE_DIR, f'fold{fold}_phase12_final.pt'))

In [ ]:
# Cell 5: Save
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, f'fold{fold}_phase12_final.pt'), map_location=DEVICE)['model_state_dict'])
model.eval()
y_true, y_prob = [], []
with torch.no_grad():
    for batch in test_loader:
        input_keys = [k for k in batch.keys() if k not in ['target', 'targets', 'participant_id']]
        outputs = model({k: batch[k].to(DEVICE) for k in input_keys})
        y_prob.extend(outputs[0]['binary_prob'].cpu().numpy())
        y_true.extend(batch.get('targets', {}).get('binary', torch.zeros(1)).cpu().numpy())

pd.DataFrame({'y_true': np.array(y_true).flatten(), 'y_prob': np.array(y_prob).flatten()}).to_csv(os.path.join(ACHIEVED_DIR, f'phase12_fold{fold}_preds.csv'), index=False)
print(f'✅ Fold {fold} Complete!')